# Insample models
## Contents:
1. Historical Var (simple and EWMA)
2. Normal and t
3. GARCH
4. MS
5. MS-GARCH

all fitted to full sample

In [1]:
import os

os.chdir("C:/Users/finle/Dev/Dissertation")
print(os.getcwd())

C:\Users\finle\Dev\Dissertation


In [2]:
from pandas import Series
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

spy = pd.read_csv("data/SPY_data.csv", index_col=0, parse_dates=True)

log_returns = spy['log_returns'].fillna(0)
type(log_returns)
spy.index
type(log_returns.index)


pandas.core.indexes.datetimes.DatetimeIndex

In [ ]:
def VaR_historical(
    ts: Series, 
    window: int,
    confidence: float =0.99
    ):
    if confidence >= 0.9999:
        raise ValueError(f"Confidence must be below 1. you entered {confidence}")
    VaR = - ts.rolling(window,min_periods=50).quantile(1-confidence)
    return VaR

VaR_hist_95 = - VaR_historical(log_returns,1000, 0.95)
VaR_hist_99 = - VaR_historical(log_returns,1000, 0.99)

fig, ax = plt.subplots()
ax.plot(log_returns)
ax.plot(VaR_hist_95, color='red', label=r"$VaR(95\%)$")
ax.plot(VaR_hist_99, color="purple", label=r"$VaR(99\%$)")
plt.legend()
plt.show()


## EWMA RISKMETRICS

In [ ]:
def ewma_var(
    ts: Series,
    window:int,
    lambda_: float,
    confidence: float
    ):
    ewma_vol = ts.ewm(alpha=1-lambda_, min_periods=50,adjust=False).std()
    mean = ts.rolling(window=window,min_periods=50).mean()
    z_score = ts.rolling(window=window, min_periods=50).quantile(1- confidence)
    
    var =  - (mean + z_score * ewma_vol)
    
    return var

ewma_95 = ewma_var(log_returns, window=500, lambda_=0.96, confidence=0.95)
ewma_99 = ewma_var(log_returns, window=500, lambda_=0.96, confidence=0.99)
fig, ax = plt.subplots()
ax.plot(log_returns)
ax.plot(-ewma_95, color='red', label=r"$VaR(95\%)$")
ax.plot(-ewma_99, color="purple", label=r"$VaR(99\%$")
plt.legend()
plt.show()

## Parametric

In [ ]:
from scipy.stats import norm

def normal_VaR(
    ts: Series, 
    confidence: float,
    window: int = 1000,
    min_periods: int = 50):
    mean = ts.rolling(window=window,min_periods=min_periods).mean()
    std = ts.rolling(window=window,min_periods=min_periods).std()
    
    Z = norm.ppf(1-confidence)
    
    VaR = mean + Z * std
    
    return VaR

parametric_VaR_95 = normal_VaR(log_returns,0.95)
parametric_VaR_99 = normal_VaR(log_returns,0.99)

print(parametric_VaR_95.describe())
print(parametric_VaR_99.describe())

fig, ax = plt.subplots()
ax.plot(log_returns)
ax.plot(parametric_VaR_95, color='red', label=r"$VaR(95\%)$")
ax.plot(parametric_VaR_99, color="purple", label=r"$VaR(99\%$")
plt.legend()
plt.show()

## GARCH

In [ ]:
from arch import arch_model

am1 = arch_model(log_returns, mean='Constant', vol='GARCH', p=1, q=1, dist='normal')
garch_normal_result = am1.fit(update_freq=5, disp='off')
print(garch_normal_result.summary())

In [ ]:
VaR_garch_n_95 = (garch_normal_result.params.mu + 
                  norm.ppf(0.05) * garch_normal_result.conditional_volatility)
VaR_garch_n_99 = (garch_normal_result.params.mu + 
                  norm.ppf(0.01) * garch_normal_result.conditional_volatility)

fig, ax = plt.subplots(figsize=(12,6))
ax.plot(log_returns, linewidth=0.8 ,alpha=0.8)
ax.plot(VaR_garch_n_95, linewidth=0.5, color="red")
ax.plot(VaR_garch_n_99, linewidth=0.5, color="purple")


In [ ]:
am2 = arch_model(log_returns, mean='Constant', vol='GARCH', p=1, q=1,o=1, dist='t')
garch_t_result = am2.fit(update_freq=5)
print(garch_t_result.summary())

In [ ]:
from scipy.stats import t

scaling = np.sqrt((garch_t_result.params.nu - 2) / garch_t_result.params.nu)

VaR_garch_t_95 = (garch_t_result.params.mu + 
                  garch_t_result.conditional_volatility * 
                  t.ppf(0.05, df= garch_t_result.params.nu) * 
                  scaling)
VaR_garch_t_99 = (garch_t_result.params.mu +
                  garch_t_result.conditional_volatility *
                  t.ppf(0.01, df= garch_t_result.params.nu) *
                  scaling)

fig, ax = plt.subplots(figsize=(12,6), dpi=600)
ax.plot(log_returns, linewidth=0.8, alpha=0.8)
ax.plot(VaR_garch_t_95, linewidth=0.5, color="red")
ax.plot(VaR_garch_t_99, linewidth=0.5, color="purple")


## EGARCH

In [ ]:
egarch = arch_model(log_returns, mean='Constant', vol='EGARCH', p=1, q=1, dist='t')
egarch_result  = egarch.fit(update_freq=5)
print(egarch_result.summary())

### markov

In [ ]:
from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression

model = MarkovRegression(log_returns, k_regimes=3, trend='c', switching_trend=True, switching_variance=True)
markov_result = model.fit()

print(markov_result.summary())

print(markov_result.regime_transition)

In [ ]:
from statsmodels.tsa.regime_switching.markov_autoregression import MarkovAutoregression

model2 = MarkovAutoregression(log_returns, k_regimes=3, order=1, trend='c', switching_ar=True, switching_trend=True, switching_variance=True)
markov_result2 = model2.fit(method="bfgs", disp=1, maxiter=1000)

print(markov_result2.summary())

In [ ]:
variances = markov_result.params.filter(like='sigma2')
high_vol = variances.argmax()

probs_filtered = markov_result.filtered_marginal_probabilities[high_vol] 


fig, ax1 = plt.subplots(figsize=(12,6))

ax1.plot(log_returns, linewidth=0.5, label='Log Returns', color='blue')
ax1.set_xlabel('Date')
ax1.set_ylabel('Log Returns', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')


ax2 = ax1.twinx()
ax2.plot(probs_filtered, label='Regime 1 Probability', color='red', alpha=0.7)
ax2.set_ylabel('Regime 1 Probability', color='red')
ax2.tick_params(axis='y', labelcolor='red')
ax2.set_ylim(0, 1) 


lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper left')

plt.title('S&P 500 Squared Log Returns and Probability of High-Volatility Regime')
plt.show()

## MS-GARCH